In [29]:
from typing import Annotated
from typing_extensions import TypedDict
import os
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
from IPython.display import Image, display
from langchain_tavily import TavilySearch
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command, interrupt
from langchain_core.tools import tool

In [30]:
load_dotenv()
llm = ChatAnthropic(model='claude-sonnet-4-6')
llm

ChatAnthropic(profile={'max_input_tokens': 200000, 'max_output_tokens': 64000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'structured_output': False}, model='claude-sonnet-4-6', max_tokens=64000, anthropic_api_url='https://api.anthropic.com', anthropic_api_key=SecretStr('**********'), model_kwargs={})

In [31]:
class State(TypedDict):
    messages:Annotated[list, add_messages] 

In [32]:
graph_builder = StateGraph(State)

@tool
def human_assistance(query:str) -> str:
    """Request Human Assistance"""
    human_response = interrupt({"query":query})
    return human_response["data"]

tool = TavilySearch(max_results=2)
tools = [tool, human_assistance]
llm_with_tools = llm.bind_tools(tools)

def chatbot(state:State):
    return {"messages":[llm_with_tools.invoke(state["messages"])]}

In [ ]:
graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition)

graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge("chatbot", END)

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    print("issue")

In [34]:
user_input = "I need expert guidance for building AI agents. Request some assistance for me"
config = {"configurable":{"thread_id":1}}

events = graph.stream({"messages":user_input}, config, stream_mode="values")

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

I need expert guidance for building AI agents. Request some assistance for me
================================== Ai Message ==================================

[{'id': 'toolu_019tY3QWAu73eamy8hy4ignC', 'caller': {'type': 'direct'}, 'input': {'query': 'I need expert guidance for building AI agents. Can you provide advice, best practices, frameworks, tools, and key considerations for designing and developing AI agents effectively?'}, 'name': 'human_assistance', 'type': 'tool_use'}]
Tool Calls:
  human_assistance (toolu_019tY3QWAu73eamy8hy4ignC)
 Call ID: toolu_019tY3QWAu73eamy8hy4ignC
  Args:
    query: I need expert guidance for building AI agents. Can you provide advice, best practices, frameworks, tools, and key considerations for designing and developing AI agents effectively?
================================== Ai Message ==================================

[{'id': 'toolu_019tY3QWAu73eamy8hy4ignC', 'cal

In [ ]:
human_response = ("Use LangGraph")

human_command = Command(resume = {"data":human_response})


events = graph.stream(human_command, config, stream_mode="values")

for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()


RuntimeError: Cannot use Command(resume=...) without checkpointer